In [1]:
import pandas as pd
import numpy as np
import torch
import importlib
from importlib import reload
import os
import sys
sys.path.append(os.path.abspath('../../')) 
from protocol import ScoreSpectra, BayesianOptimization,GaussianProcess, PlotGP, MaraData,TxtToCsvConverter

import itertools
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import re
from scipy.stats import qmc
import random

dtype = torch.float64

In [2]:
# PbI2:BAAc:MAI = 1 : (2/n) : [(n+1)/n]
theoretical_N2 = {'Anneal Time': 10, # min
                  'Temperature': 150, # C
                  'R BAAc': 1.00,  
                  'R MAI': 1.50,
                  'R PbI2': 1,
                  }

# PbI2:BAAc:MAI = 1 : (2/n) : [(n+1)/n]
theoretical_N1 = {'Anneal Time': 10, # min
                  'Temperature': 150, # C
                  'R BAAc': 2.00,  
                  'R MAI': 2.00,
                  'R PbI2': 1,
                  }

# PbI2:BAAc:MAI = 1 : (2/n) : [(n+1)/n]
theoretical_N3 = {'Anneal Time': 10, # min
                  'Temperature': 150, # C
                  'R BAAc': 0.667,  # aprox to 0.675
                  'R MAI': 1.333, # aprox to 1.325
                  'R PbI2': 1,
                  }

theoretical_N4 = {'Anneal Time': 10, # min
                  'Temperature': 95, # C
                  'R BAAc': 0.5,  
                  'R MAI': 1.25,
                  'R PbI2': 1,
                  }

In [3]:
N1_data = pd.read_csv('../N1_Campaign/Data/N1Campaign_data.csv')
N2_data = pd.read_csv('../N2_Campaign/Data/N2Campaign_data.csv')
N3_data = pd.read_csv('../N3_Campaign/Data/N3Campaign_data.csv')

In [4]:
# 2D GP of the stoicheometry
def data_prep(data, QW_num):
    # Past experimental data loading
    previous_data = data[['Anneal Time','Temperature','R BAAc', 'R MAI', 'R PbI2', f'{QW_num}']]
    data_df = previous_data[['Anneal Time','R PbI2','R BAAc', 'R MAI', 'Temperature']][::2].reset_index(drop=True)

    QW_mean = previous_data[f'{QW_num}'].groupby(previous_data.index // 2).mean().reset_index(drop=True)
    QW_var = previous_data[f'{QW_num}'].groupby(previous_data.index // 2).var().reset_index(drop=True)
    data_df[f'{QW_num} mean'] = QW_mean
    data_df[f'{QW_num} var'] = QW_var

    return data_df

def gp_stoich(data_df, QW_num, bounds):
    dtype = torch.float64
    original_bounds = torch.tensor([[0.40, 1.1],[1.2, 1.6]], dtype=dtype) #  R BAAc, R MAI

    # Set up of the traning data
    train_X = data_df[['R BAAc', 'R MAI']].to_numpy()
    train_y = data_df[f'{QW_num} mean'].to_numpy()
    train_yvar = data_df[f'{QW_num} var'].to_numpy() # Use the mean off {QW_num} for training

    gaussian = GaussianProcess(train_X, train_y, train_yvar=train_yvar,bounds = original_bounds.T)
    gaussian.fit()

    gaussian.evaluate()

def gp_process(data_df, QW_num):
    dtype = torch.float64
    original_bounds = torch.tensor([[5, 60], [60, 150]], dtype=dtype) #  R BAAc, R MAI

    # Set up of the traning data
    train_X = data_df[['Anneal Time','Temperature']].to_numpy()
    train_y = data_df[f'{QW_num} mean'].to_numpy()
    train_yvar = data_df[f'{QW_num} var'].to_numpy() # Use the mean off {QW_num} for training

    gaussian = GaussianProcess(train_X, train_y, train_yvar=train_yvar,bounds = original_bounds.T)
    gaussian.fit()




In [ ]:
from matplotlib import animation
from IPython.display import HTML
import matplotlib as mpl
import matplotlib, IPython
import itertools
from matplotlib.animation import FFMpegWriter


def plot_gp_surface(data, bounds, qw_col, title_suffix='', train_type = 'stoich'):
    if train_type == 'stoich':
        columns = ['R BAAc', 'R MAI']
    elif train_type == 'process':
        columns = ['Anneal Time', 'Temperature']

    bounds = bounds.T

    data['round'] = data['files'].str.extract(r'R(\d+)', expand=False).astype(int)

    # optional: inspect counts per round to verify R0 has 18 points
    print("counts per round:\n", data['round'].value_counts().sort_index())

    groups = [g for _, g in data.groupby('round', sort=True)]

    data_train = data_prep(data, qw_col)

    x = np.linspace(0, 1, 50)
    y = np.linspace(0, 1, 50)
    X, Y = np.meshgrid(x, y)
    XY = np.c_[X.ravel(), Y.ravel()]

    fig, ax = plt.subplots(figsize=(6, 5))
    start = 0
    artists = []
    last_cf = None
    for idx, g in enumerate(groups):
        start += len(g)//2 

        train_X = data_train[columns][:start].to_numpy()
        train_y = data_train[f'{qw_col} mean'][:start].to_numpy()
        train_yvar = data_train[f'{qw_col} var'][:start].to_numpy()

        print(train_X.shape, train_y.shape, train_yvar.shape)

        gp = GaussianProcess(train_X, train_y, train_yvar=train_yvar,bounds=bounds)
        gp.fit()

        # Generate meshgrid for contour
        z = gp.evaluate(XY)[0].reshape(X.shape)

        frame_artists = []
        
        cf = ax.contourf(X, Y, z, levels=50, cmap='viridis', vmin=0, vmax=0.5)
        frame_artists.extend(cf.collections)  

        if idx == 0:
            pts =  ax.scatter(
            (train_X[:, 0] - bounds[0, 0].item()) / (bounds[1, 0].item() - bounds[0, 0].item()),
            (train_X[:, 1] - bounds[0, 1].item()) / (bounds[1, 1].item() - bounds[0, 1].item()),
            marker = 'x', color='r', s=80, alpha=0.9)
            frame_artists.append(pts)
        else:
            # Optional: overlay training points
            pts =  ax.scatter(
                (train_X[:, 0] - bounds[0, 0].item()) / (bounds[1, 0].item() - bounds[0, 0].item()),
                (train_X[:, 1] - bounds[0, 1].item()) / (bounds[1, 1].item() - bounds[0, 1].item()),
                marker = '.',color='k', s=80, alpha=0.5
            )
            ax.legend()
            frame_artists.append(pts)

        artists.append(frame_artists)
        last_cf = cf
        frame_artists.append(pts)
        
    # plt.colorbar(cf, ax=ax, label='Predicted QW 1')

    if last_cf is not None:
        plt.colorbar(last_cf, ax=ax, label=f'Predicted {qw_col}')
    
    # Set ticks and labels for unnormalized axis
    xticks = np.linspace(0, 1, 5)
    xticklabels = np.round(
        bounds[0, 0].item() + xticks * (bounds[1, 0].item() - bounds[0, 0].item()), 2
    )
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)

    yticks = np.linspace(0, 1, 5)
    yticklabels = np.round(
        bounds[0, 1].item() + yticks * (bounds[1, 1].item() - bounds[0, 1].item()), 2
    )
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)

    ax.set_xlabel('R BAAc')
    ax.set_ylabel('R MAI')
    ax.set_title(title_suffix)

    


In [13]:
N1_bounds = torch.tensor([[0.6, 2.5],[1.3, 2.5]], dtype=dtype) #  R BAAc, R MAI

In [7]:
N1_bounds = torch.tensor([[5, 60], [60, 150]], dtype=dtype) 

In [8]:
N2_bounds = torch.tensor([[0.40, 1.8],[1.2, 1.8]], dtype=dtype) #  R BAAc, R MAI

In [9]:
N2_bounds = torch.tensor([[5, 60], [60, 150]], dtype=dtype) #  R BAAc, R MAI



In [10]:
# N2_bounds = torch.tensor([[0.40, 1.8],[1.2, 1.8]], dtype=dtype) #  R BAAc, R MAI
# # Select data for this time/temp
# N2_data_train = data_prep(N2_data, 'QW 2')
# train_X = N2_data_train[['R BAAc', 'R MAI']].to_numpy()
# train_y = N2_data_train['QW 2 mean'].to_numpy()
# train_yvar = N2_data_train['QW 2 var'].to_numpy()
# plot_gp_surface(train_X, train_y, N2_bounds, title_suffix='GP Predicted Surface for QW 2')

# N2_bounds = torch.tensor([[5, 60], [60, 150]], dtype=dtype) #  R BAAc, R MAI
# train_X = N2_data_train[['Anneal Time', 'Temperature']].to_numpy()
# train_y = N2_data_train['QW 2 mean'].to_numpy()
# train_yvar = N2_data_train['QW 2 var'].to_numpy()
# plot_gp_surface(train_X, train_y, N2_bounds, title_suffix='GP Predicted Surface for QW 2')


In [11]:
N3_bounds = torch.tensor([[0.40, 1.1],[1.2, 1.6]], dtype=dtype) #  R BAAc, R MAI


In [12]:
N3_bounds = torch.tensor([[5, 60], [60, 150]], dtype=dtype) #  R BAAc, R MAI

